# PrivTrace

Adapter Code to evaluate PrivTrace on our datasets

In [ ]:
import logging
from pathlib import Path

import numpy as np

from privtrajgen import config
from privtrajgen.config import Datasets
from privtrajgen.datasets import get_difftraj_dataset, denormalise_by_dataset
from privtrajgen.datasets.privtrace_adapter import transform_data_from_difftraj_to_privtrace, \
    transform_data_from_privtrace_to_difftraj
from privtrajgen.evaluation.split_dataset import get_train_set
from privtrajgen.utils import printc, configure_root_loger


In [ ]:
%load_ext autoreload
%autoreload 2

log = configure_root_loger(logging.INFO)

## Convert our datasets to PrivTrace format

In [ ]:
# Constants
OUTPUT_DIR = Path(config.BASE_DIR) / "PrivTrace" / "datasets"
printc("Output Directory:", OUTPUT_DIR)
N_RUNS = 5

In [ ]:
# Get the GeoLife dataset
datasets = [Datasets.PORTO, Datasets.GEOLIFE]
full_datasets = dict()
for name in datasets:
    ds_full, _ = get_difftraj_dataset(name, False, as_numpy=True) # 1 is the conditional information which we can ignore
    # PrivTrace expects de-normalised data! Denormalisation function works on (n_trajectories, n_points, 2) instead of (n_trajectories, 2, n_points) --> Transpose accordingly
    ds_full = denormalise_by_dataset(data=ds_full.transpose(0, 2, 1), dataset=name).transpose(0, 2, 1)
    full_datasets[name] = ds_full
    printc("Dataset Name:", name)
    printc("Dataset Type:", type(ds_full))
    printc("Dataset Dimensions:", ds_full.shape)
    dataset = get_train_set(name, 0, ds_full, n_splits=N_RUNS)
    printc("Dataset Type:", type(dataset))
    printc("Dataset Dimensions:", dataset.shape)
    printc("Value Range Lon:", dataset[:, 0].min(), dataset[:, 0].max())
    printc("Value Range Lat:", dataset[:, 1].min(), dataset[:, 1].max())
    printc("Number of Points:", f"{dataset.shape[0] * dataset.shape[2]:,}")
    if dataset.shape[0] > 200_000:
        # We will subsample later
        printc("Number of Points for 200k Trajectories:", f"{200_000 * dataset.shape[2]:,}")
    print('#' * 80)

In [ ]:
# Get the dataset
dataset = Datasets.GEOLIFE
fold = 0
geolife = get_train_set(dataset, fold, full_datasets[dataset], n_splits=N_RUNS)
# Define the output path
path = OUTPUT_DIR / f"test.dat"
# Transform the data
transform_data_from_difftraj_to_privtrace(geolife, path)

In [ ]:
# Load the file we just wrote and compare it to the original
path = OUTPUT_DIR / f"test.dat"
# Load the data
loaded_data = transform_data_from_privtrace_to_difftraj(path)
# Compare the data
print("Loaded data shape:", loaded_data.shape)
print("Original data shape:", geolife.shape)

if not np.allclose(loaded_data, geolife, rtol=1e-5, atol=1e-7):
    raise ValueError("Loaded data does not match original data.")
else:
    print("Success: Loaded data matches the original data!")

In [ ]:
# Lastly load both test.dat and GEOLIFE_0.dat and compare them to make sure the process is deterministic
loaded_data2 = transform_data_from_privtrace_to_difftraj(OUTPUT_DIR / "GEOLIFE_0.dat")
# Compare the data
printc("Original data shape:", loaded_data.shape)
printc("Loaded data shape:", loaded_data2.shape)
if not np.allclose(loaded_data, loaded_data2, rtol=1e-5, atol=1e-7):
    raise ValueError("Loaded data does not match original data.")
else:
    printc("Data matches:", "✅")

In [ ]:
for dataset in datasets:
    for fold in range(N_RUNS):
        # Define the output path
        path = OUTPUT_DIR / f"{dataset.name}_{fold}.dat"
        # Check if the file already exists - if so, write a message and skip
        if path.exists():
            printc("File already exists:", path)
            printc("Skipping dataset:", dataset, fold)
            continue
        # Get the dataset
        training_set = get_train_set(dataset, fold, full_datasets[dataset], n_splits=N_RUNS)
        # If there are more than 200'000 trajectories, randomly select 200'000 trajectories due to excessive runtime and print a warning.
        if training_set.shape[0] > 200_000:
            log.warning(f"Dataset {dataset}#{fold} has more than 200'000 trajectories. Randomly selecting 200'000 trajectories.")
            random_indices = np.random.choice(training_set.shape[0], size=200_000, replace=False)
            # Get the random trajectories
            training_set = training_set[random_indices]
        # Transform the data
        transform_data_from_difftraj_to_privtrace(training_set, path)
        # Print Confirmation
        printc("Transformed dataset:", dataset, fold)

## Testing

In [ ]:
# Randomly select 3'000 trajectories from GEOLIFE and save them to `${BASE_DIR}/PrivTrace/outputs/GEOLIFE_e1000.0_01_output.dat`
# Load the data
dataset = Datasets.GEOLIFE
fold = 0
training_set = get_train_set(dataset, fold, full_datasets[dataset], n_splits=N_RUNS)
# Get 3'000 random trajectories
random_indices = np.random.choice(training_set.shape[0], size=3000, replace=False)
# Get the random trajectories
random_trajectories = training_set[random_indices]
# Define the output path
path = Path(config.BASE_DIR) / "PrivTrace" / "outputs" / f"GEOLIFE_e1000.0_01_output.dat"
# Transform the data
transform_data_from_difftraj_to_privtrace(random_trajectories, path)